<a href="https://colab.research.google.com/github/hoangnguyen3101/Application-algorithms/blob/main/Lab3_Tong_hop_Greedy_Divide_Conquer_Large_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bài thực hành 3: Luyện tập tổng hợp Tham lam và Chia để trị trên dữ liệu lớn

Trong hai bài Lab trước, chúng ta đã học riêng:

- **thuật toán tham lam**: ra quyết định cục bộ theo một tiêu chí lựa chọn;
- **chia để trị**: chia bài toán thành các bài toán con, giải từng phần và ghép kết quả.

Bài Lab này không lặp lại các bài toán quen thuộc như đổi tiền, lập lịch hoạt động, ba lô phân số hoặc Merge Sort đơn lẻ. Thay vào đó, bài Lab tập trung vào một khuôn mẫu thường gặp trong thực tế:

> Dùng **chia để trị** để tiền xử lý dữ liệu lớn, sau đó dùng **tham lam** để ra quyết định nhanh.

Ta sẽ thực hành trên hai bài toán mới:

1. **Phủ một đoạn bằng số khoảng ít nhất**.
2. **Xếp lịch công việc có deadline và lợi nhuận**.

Cả hai bài toán đều có dữ liệu sinh ngẫu nhiên với kích thước mặc định khoảng vài chục nghìn phần tử, giúp sinh viên thấy rõ hơn vai trò của độ phức tạp thuật toán.

## 1. Mục tiêu học tập

Sau bài thực hành này, sinh viên có thể:

1. Nhận diện được tình huống cần kết hợp **sắp xếp chia để trị** và **lựa chọn tham lam**.
2. Cài đặt được một phiên bản Merge Sort tổng quát theo khóa sắp xếp.
3. Giải bài toán **phủ đoạn bằng số khoảng ít nhất** bằng chiến lược tham lam.
4. Giải bài toán **xếp lịch công việc có deadline và lợi nhuận** bằng tham lam kết hợp cấu trúc Union-Find.
5. Kiểm chứng thuật toán tham lam trên dữ liệu nhỏ bằng vét cạn.
6. Đo thời gian chạy trên dữ liệu lớn và nhận xét xu hướng tăng trưởng.

## 2. Chuẩn bị môi trường

Notebook chỉ dùng các thư viện phổ biến: `numpy` và `matplotlib`.

Cell dưới đây cài đặt thư viện theo cách an toàn, tránh dùng lệnh `!pip install` bị thụt lề sai trong notebook.

In [ ]:
import sys
import subprocess
import importlib

packages = ["numpy", "matplotlib"]

for pkg in packages:
    try:
        importlib.import_module(pkg)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Cài đặt/kiểm tra thư viện hoàn tất.")

## 3. Import thư viện và thiết lập seed

In [ ]:
import random
import time
from dataclasses import dataclass
from itertools import combinations

import numpy as np
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Seed:", SEED)

## 4. Khuôn mẫu kết hợp: chia để trị + tham lam

Trong nhiều bài toán tối ưu rời rạc, thuật toán tham lam không làm việc trực tiếp trên dữ liệu thô. Trước khi chọn quyết định cục bộ, ta thường cần sắp xếp dữ liệu theo một tiêu chí nhất định.

Khuôn mẫu tổng quát:

1. **Tiền xử lý bằng chia để trị**  
   Ví dụ: dùng Merge Sort hoặc Quick Sort để sắp xếp dữ liệu theo khóa phù hợp.

2. **Duyệt tuyến tính và ra quyết định tham lam**  
   Sau khi dữ liệu đã có thứ tự, thuật toán chỉ cần quét một lần và chọn phương án tốt nhất tại từng bước.

Nếu bước sắp xếp mất $O(n \log n)$ và bước tham lam mất $O(n)$, độ phức tạp tổng thể thường là:

$$
O(n \log n) + O(n) = O(n \log n).
$$

Điểm quan trọng là: **chia để trị không trực tiếp đưa ra nghiệm tối ưu**, nhưng nó tạo ra cấu trúc dữ liệu giúp lựa chọn tham lam trở nên đúng và hiệu quả.

## 5. Cài đặt Merge Sort tổng quát

Ở Lab 2, Merge Sort thường được trình bày để sắp xếp mảng số. Trong bài này, ta cần sắp xếp các đối tượng phức tạp hơn, ví dụ:

- khoảng $[l_i, r_i]$ theo điểm bắt đầu;
- công việc theo lợi nhuận giảm dần;
- nếu bằng nhau thì xét thêm deadline hoặc chỉ số công việc.

Do đó, ta cài đặt hàm `merge_sort_by_key(items, key)` tương tự `sorted(items, key=...)`, nhưng dùng ý tưởng chia để trị.

In [ ]:
def merge_sort_by_key(items, key=lambda x: x):
    """
    Sắp xếp danh sách items theo khóa key bằng Merge Sort.

    Hàm trả về một danh sách mới, không làm thay đổi danh sách ban đầu.
    Cách cài đặt dùng kỹ thuật decorate-sort-undecorate:
    - lưu trước key(item) để tránh tính lại nhiều lần;
    - thêm chỉ số ban đầu để giữ tính ổn định khi hai khóa bằng nhau.
    """
    decorated = [(key(item), idx, item) for idx, item in enumerate(items)]

    def _merge_sort(a):
        n = len(a)
        if n <= 1:
            return a

        mid = n // 2
        left = _merge_sort(a[:mid])
        right = _merge_sort(a[mid:])

        merged = []
        i = j = 0

        while i < len(left) and j < len(right):
            # So sánh theo khóa; nếu khóa bằng nhau, giữ thứ tự ban đầu.
            if (left[i][0] < right[j][0]) or (
                left[i][0] == right[j][0] and left[i][1] <= right[j][1]
            ):
                merged.append(left[i])
                i += 1
            else:
                merged.append(right[j])
                j += 1

        merged.extend(left[i:])
        merged.extend(right[j:])
        return merged

    return [item for _, _, item in _merge_sort(decorated)]


# Kiểm tra nhanh
numbers = [5, 1, 8, 3, 3, 2]
print("Trước khi sắp xếp:", numbers)
print("Sau khi sắp xếp :", merge_sort_by_key(numbers))

### Nhận xét

Trong thực tế, Python đã có hàm `sorted()` được tối ưu rất tốt. Tuy nhiên, việc tự cài đặt Merge Sort trong bài Lab này giúp sinh viên hiểu rõ vai trò của bước **chia để trị** trong các thuật toán tổng hợp.

Ở các phần sau, ta vẫn dùng `merge_sort_by_key` để giữ đúng mục tiêu học tập.

## 6. Bài toán 1: Phủ một đoạn bằng số khoảng ít nhất

### 6.1. Phát biểu bài toán

Cho một đoạn mục tiêu $[L, R]$ và một tập các khoảng:

$$
I_1, I_2, \ldots, I_n,
$$

trong đó mỗi khoảng có dạng:

$$
I_i = [l_i, r_i].
$$

Mục tiêu là chọn ra **ít khoảng nhất** sao cho hợp của các khoảng được chọn phủ toàn bộ đoạn $[L, R]$.

Ví dụ, cần phủ đoạn $[0, 10]$ bằng các khoảng con. Ta muốn chọn ít khoảng nhất nhưng vẫn đảm bảo không có điểm nào trong $[0,10]$ bị bỏ trống.

### 6.2. Ý tưởng tham lam

Giả sử ta đang phủ đến vị trí hiện tại là `current`.

Trong tất cả các khoảng có điểm bắt đầu không vượt quá `current`, tức là:

$$
l_i \le current,
$$

ta chọn khoảng có điểm kết thúc xa nhất.

Trực giác:

- mọi khoảng được chọn tiếp theo đều phải bắt đầu trước hoặc tại `current`;
- trong các khoảng hợp lệ đó, chọn khoảng vươn xa nhất sẽ không làm nghiệm tệ hơn;
- sau khi chọn xong, ta cập nhật `current` thành điểm kết thúc của khoảng vừa chọn.

Để làm được điều này hiệu quả, ta sắp xếp các khoảng theo điểm bắt đầu tăng dần. Đây chính là bước dùng **chia để trị**.

In [ ]:
@dataclass
class Interval:
    start: int
    end: int
    name: str = ""


def generate_interval_cover_instance(
    n=50_000,
    target_start=0,
    target_end=10_000,
    seed=SEED
):
    """
    Sinh dữ liệu lớn cho bài toán phủ đoạn.

    Hàm đảm bảo tồn tại ít nhất một cách phủ [target_start, target_end]
    bằng cách tạo trước một chuỗi khoảng liên tiếp, sau đó thêm nhiều khoảng nhiễu.
    """
    rng = random.Random(seed)
    intervals = []

    # Tạo một chuỗi khoảng đảm bảo phủ được đoạn mục tiêu.
    current = target_start
    idx = 0
    while current < target_end:
        start = max(target_start, current - rng.randint(0, 100))
        length = rng.randint(120, 600)
        end = min(target_end, current + length)

        if end <= current:
            end = min(target_end, current + 1)

        intervals.append(Interval(start, end, f"chain_{idx}"))
        current = end
        idx += 1

    # Thêm các khoảng ngẫu nhiên để dữ liệu lớn và thực tế hơn.
    while len(intervals) < n:
        start = rng.randint(target_start, target_end - 1)
        length = rng.randint(10, 800)
        end = min(target_end, start + length)
        intervals.append(Interval(start, end, f"noise_{len(intervals)}"))

    rng.shuffle(intervals)
    return intervals


def cover_interval_greedy(intervals, target_start, target_end):
    """
    Thuật toán tham lam phủ đoạn bằng số khoảng ít nhất.

    Bước 1: sắp xếp các khoảng theo start tăng dần;
            nếu start bằng nhau, khoảng có end lớn hơn được xét trước.
    Bước 2: quét từ trái sang phải và luôn chọn khoảng vươn xa nhất.
    """
    sorted_intervals = merge_sort_by_key(
        intervals,
        key=lambda it: (it.start, -it.end)
    )

    selected = []
    current = target_start
    i = 0
    n = len(sorted_intervals)

    while current < target_end:
        best_interval = None
        best_end = current

        # Xét tất cả khoảng có start <= current.
        while i < n and sorted_intervals[i].start <= current:
            if sorted_intervals[i].end > best_end:
                best_interval = sorted_intervals[i]
                best_end = sorted_intervals[i].end
            i += 1

        # Không có khoảng nào nối tiếp được phần đã phủ.
        if best_interval is None:
            return None, sorted_intervals

        selected.append(best_interval)
        current = best_end

    return selected, sorted_intervals

### 6.3. Chạy trên dữ liệu lớn

Dữ liệu mặc định gồm **50.000 khoảng** và đoạn cần phủ là $[0, 10000]$.

Nếu máy yếu, có thể giảm `N_INTERVALS` xuống `10000` hoặc `20000`. Nếu muốn thử dữ liệu lớn hơn, có thể tăng lên `100000` hoặc `200000`.

In [ ]:
N_INTERVALS = 50_000
TARGET_START = 0
TARGET_END = 10_000

intervals = generate_interval_cover_instance(
    n=N_INTERVALS,
    target_start=TARGET_START,
    target_end=TARGET_END,
    seed=SEED
)

start_time = time.perf_counter()
selected_intervals, sorted_intervals = cover_interval_greedy(
    intervals,
    TARGET_START,
    TARGET_END
)
elapsed = time.perf_counter() - start_time

print(f"Số khoảng đầu vào: {len(intervals):,}")
print(f"Thời gian chạy: {elapsed:.4f} giây")

if selected_intervals is None:
    print("Không thể phủ toàn bộ đoạn mục tiêu.")
else:
    print(f"Số khoảng được chọn: {len(selected_intervals)}")
    print("Một vài khoảng được chọn:")
    for it in selected_intervals[:10]:
        print(it)

### 6.4. Trực quan hóa nghiệm

Với dữ liệu 50.000 khoảng, ta không nên vẽ tất cả vì hình sẽ rối. Cell dưới đây chỉ vẽ:

- một mẫu nhỏ các khoảng không được chọn;
- toàn bộ các khoảng được chọn bởi thuật toán tham lam.

In [ ]:
def plot_interval_cover(intervals, selected, target_start, target_end, sample_size=80, seed=SEED):
    rng = random.Random(seed)

    selected_set = set((it.start, it.end, it.name) for it in selected)
    non_selected = [
        it for it in intervals
        if (it.start, it.end, it.name) not in selected_set
    ]

    sample = rng.sample(non_selected, min(sample_size, len(non_selected)))

    plt.figure(figsize=(10, 5))

    # Vẽ một số khoảng không được chọn.
    for idx, it in enumerate(sample):
        plt.hlines(
            y=idx,
            xmin=it.start,
            xmax=it.end,
            linewidth=1,
            alpha=0.35
        )

    offset = len(sample) + 3

    # Vẽ các khoảng được chọn.
    for j, it in enumerate(selected):
        plt.hlines(
            y=offset + j,
            xmin=it.start,
            xmax=it.end,
            linewidth=3
        )
        plt.text(it.start, offset + j + 0.15, it.name, fontsize=8)

    # Vẽ đoạn mục tiêu.
    plt.hlines(
        y=offset + len(selected) + 2,
        xmin=target_start,
        xmax=target_end,
        linewidth=4
    )
    plt.text(target_start, offset + len(selected) + 2.5, "Đoạn mục tiêu", fontsize=10)

    plt.xlabel("Vị trí trên trục số")
    plt.ylabel("Các khoảng")
    plt.title("Phủ đoạn bằng thuật toán tham lam")
    plt.grid(True, axis="x", alpha=0.3)
    plt.show()


if selected_intervals is not None:
    plot_interval_cover(intervals, selected_intervals, TARGET_START, TARGET_END)

### 6.5. Kiểm chứng bằng vét cạn trên dữ liệu nhỏ

Với dữ liệu lớn, ta không thể vét cạn vì số tập con là $2^n$. Tuy nhiên, trên dữ liệu nhỏ, có thể dùng vét cạn để kiểm tra xem nghiệm tham lam có tối ưu hay không.

Ý tưởng kiểm chứng:

1. sinh một bộ dữ liệu nhỏ;
2. chạy thuật toán tham lam;
3. thử tất cả các tập con để tìm nghiệm ít khoảng nhất;
4. so sánh số khoảng.

In [ ]:
def is_cover(subset, target_start, target_end):
    """
    Kiểm tra subset các khoảng có phủ toàn bộ [target_start, target_end] hay không.
    """
    if not subset:
        return False

    sorted_subset = merge_sort_by_key(subset, key=lambda it: (it.start, it.end))

    current = target_start
    for it in sorted_subset:
        if it.start > current:
            return False
        if it.end > current:
            current = it.end
        if current >= target_end:
            return True

    return current >= target_end


def brute_force_min_interval_cover(intervals, target_start, target_end):
    """
    Vét cạn tìm số khoảng ít nhất để phủ đoạn.
    Chỉ dùng cho n rất nhỏ.
    """
    n = len(intervals)

    for k in range(1, n + 1):
        for subset in combinations(intervals, k):
            if is_cover(subset, target_start, target_end):
                return list(subset)

    return None


small_intervals = [
    Interval(0, 3, "A"),
    Interval(0, 2, "B"),
    Interval(2, 5, "C"),
    Interval(3, 7, "D"),
    Interval(5, 10, "E"),
    Interval(7, 10, "F"),
    Interval(1, 6, "G"),
    Interval(6, 10, "H"),
]

greedy_small, _ = cover_interval_greedy(small_intervals, 0, 10)
optimal_small = brute_force_min_interval_cover(small_intervals, 0, 10)

print("Số khoảng tham lam:", len(greedy_small))
print("Số khoảng tối ưu :", len(optimal_small))
print("Nghiệm tham lam:", greedy_small)
print("Nghiệm tối ưu :", optimal_small)

## 7. Bài toán 2: Xếp lịch công việc có deadline và lợi nhuận

### 7.1. Phát biểu bài toán

Có $n$ công việc. Mỗi công việc $j$ có:

- deadline $d_j$;
- lợi nhuận $p_j$;
- thời lượng xử lý bằng 1 đơn vị thời gian.

Mỗi thời điểm chỉ làm được nhiều nhất một công việc. Một công việc đem lại lợi nhuận nếu được hoàn thành không muộn hơn deadline của nó.

Mục tiêu là chọn và xếp lịch một số công việc sao cho tổng lợi nhuận lớn nhất.

### 7.2. Ý tưởng tham lam

Chiến lược tham lam kinh điển:

1. Sắp xếp các công việc theo lợi nhuận giảm dần.
2. Xét từng công việc theo thứ tự đó.
3. Nếu còn slot thời gian trống không muộn hơn deadline, xếp công việc vào **slot muộn nhất có thể**.

Tại sao chọn slot muộn nhất?

- Giữ lại các slot sớm cho những công việc có deadline gấp hơn.
- Không làm giảm lợi nhuận hiện tại vì mỗi công việc có thời lượng 1.
- Kết hợp với thứ tự lợi nhuận giảm dần, đây là một chiến lược tham lam tối ưu cho biến thể công việc đơn vị thời gian.

Bước sắp xếp dùng Merge Sort. Bước tìm slot trống được tối ưu bằng cấu trúc **Union-Find**.

In [ ]:
@dataclass
class Job:
    job_id: int
    deadline: int
    profit: int


def generate_jobs(n=50_000, max_deadline=10_000, seed=SEED):
    """
    Sinh dữ liệu lớn cho bài toán xếp lịch công việc.

    max_deadline nhỏ hơn n để tạo tình huống cạnh tranh:
    không thể chọn tất cả công việc, thuật toán phải ưu tiên công việc lợi nhuận cao.
    """
    rng = random.Random(seed)
    jobs = []

    for job_id in range(n):
        deadline = rng.randint(1, max_deadline)
        profit = rng.randint(10, 1000)
        jobs.append(Job(job_id, deadline, profit))

    return jobs


def schedule_jobs_greedy(jobs, max_deadline=None):
    """
    Xếp lịch công việc bằng tham lam + Union-Find.

    Ý nghĩa parent[t]:
    - find(t) trả về slot trống muộn nhất không vượt quá t.
    - nếu slot t đã dùng, ta nối t về t-1.
    """
    if len(jobs) == 0:
        return [], 0, [], []

    D = max(job.deadline for job in jobs)
    if max_deadline is not None:
        D = min(D, max_deadline)

    # Sắp xếp lợi nhuận giảm dần bằng cách dùng khóa -profit.
    sorted_jobs = merge_sort_by_key(
        jobs,
        key=lambda job: (-job.profit, job.deadline, job.job_id)
    )

    parent = list(range(D + 1))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    schedule = [None] * (D + 1)
    total_profit = 0

    for job in sorted_jobs:
        latest_slot = find(min(job.deadline, D))

        if latest_slot > 0:
            schedule[latest_slot] = job
            total_profit += job.profit

            # Slot latest_slot đã dùng, slot trống tiếp theo không vượt quá nó là latest_slot - 1.
            parent[latest_slot] = find(latest_slot - 1)

    selected_jobs = [job for job in schedule[1:] if job is not None]
    return selected_jobs, total_profit, schedule, sorted_jobs

### 7.3. Chạy trên dữ liệu lớn

Dữ liệu mặc định gồm **50.000 công việc** và tối đa **10.000 slot thời gian**.

Vì số slot nhỏ hơn số công việc, thuật toán buộc phải chọn một tập con công việc có lợi nhuận cao.

In [ ]:
N_JOBS = 50_000
MAX_DEADLINE = 10_000

jobs = generate_jobs(
    n=N_JOBS,
    max_deadline=MAX_DEADLINE,
    seed=SEED
)

start_time = time.perf_counter()
selected_jobs, total_profit, schedule, sorted_jobs = schedule_jobs_greedy(
    jobs,
    max_deadline=MAX_DEADLINE
)
elapsed = time.perf_counter() - start_time

print(f"Số công việc đầu vào: {len(jobs):,}")
print(f"Số slot thời gian tối đa: {MAX_DEADLINE:,}")
print(f"Số công việc được xếp lịch: {len(selected_jobs):,}")
print(f"Tổng lợi nhuận: {total_profit:,}")
print(f"Thời gian chạy: {elapsed:.4f} giây")

print("\nMột vài slot cuối trong lịch:")
count = 0
for t in range(len(schedule) - 1, 0, -1):
    if schedule[t] is not None:
        print(f"Slot {t}: {schedule[t]}")
        count += 1
    if count == 10:
        break

### 7.4. Trực quan hóa phân bố lợi nhuận

Ta so sánh phân bố lợi nhuận của:

- toàn bộ công việc đầu vào;
- các công việc được thuật toán chọn.

Nếu thuật toán hoạt động đúng trực giác, nhóm được chọn thường có lợi nhuận cao hơn đáng kể.

In [ ]:
all_profits = [job.profit for job in jobs]
selected_profits = [job.profit for job in selected_jobs]

plt.figure(figsize=(9, 4))
plt.hist(all_profits, bins=30, alpha=0.5, label="Tất cả công việc")
plt.hist(selected_profits, bins=30, alpha=0.7, label="Công việc được chọn")
plt.xlabel("Lợi nhuận")
plt.ylabel("Số lượng")
plt.title("Phân bố lợi nhuận trước và sau khi chọn")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Lợi nhuận trung bình của tất cả công việc:", round(np.mean(all_profits), 2))
print("Lợi nhuận trung bình của công việc được chọn:", round(np.mean(selected_profits), 2))

### 7.5. Kiểm chứng bằng vét cạn trên dữ liệu nhỏ

Với $n=50.000$, không thể vét cạn. Nhưng với $n$ nhỏ, ta có thể kiểm tra tất cả tập con để xác nhận nghiệm tham lam.

Một tập công việc là khả thi nếu sau khi sắp theo deadline tăng dần, công việc thứ $t$ có deadline ít nhất là $t$.

In [ ]:
def is_feasible_job_set(job_subset):
    """
    Kiểm tra một tập công việc có thể xếp lịch trước deadline hay không.
    Vì mỗi công việc có thời lượng 1, ta sắp theo deadline tăng dần.
    """
    ordered = merge_sort_by_key(job_subset, key=lambda job: job.deadline)

    for t, job in enumerate(ordered, start=1):
        if t > job.deadline:
            return False

    return True


def brute_force_job_scheduling(jobs):
    """
    Vét cạn tìm tập công việc có tổng lợi nhuận lớn nhất.
    Chỉ dùng cho n nhỏ.
    """
    n = len(jobs)
    best_subset = []
    best_profit = 0

    for k in range(1, n + 1):
        for subset in combinations(jobs, k):
            if is_feasible_job_set(subset):
                profit = sum(job.profit for job in subset)
                if profit > best_profit:
                    best_profit = profit
                    best_subset = list(subset)

    return best_subset, best_profit


small_jobs = [
    Job(0, 2, 100),
    Job(1, 1, 19),
    Job(2, 2, 27),
    Job(3, 1, 25),
    Job(4, 3, 15),
    Job(5, 3, 60),
    Job(6, 2, 45),
    Job(7, 1, 30),
]

greedy_jobs, greedy_profit, _, _ = schedule_jobs_greedy(small_jobs)
optimal_jobs, optimal_profit = brute_force_job_scheduling(small_jobs)

print("Lợi nhuận tham lam:", greedy_profit)
print("Lợi nhuận tối ưu :", optimal_profit)
print("Nghiệm tham lam:", greedy_jobs)
print("Nghiệm tối ưu :", optimal_jobs)

## 8. Thí nghiệm: ảnh hưởng của kích thước dữ liệu

Phần này đo thời gian chạy của hai thuật toán khi kích thước dữ liệu tăng dần.

Lưu ý:

- Kết quả thời gian có thể khác nhau tùy máy.
- Đường cong không nhất thiết mượt tuyệt đối vì còn phụ thuộc vào bộ nhớ, hệ điều hành và trạng thái máy.
- Mục tiêu chính là quan sát xu hướng gần với $O(n \log n)$ do bước sắp xếp chi phối.

In [ ]:
def time_function(func, *args, repeats=3):
    times = []

    for _ in range(repeats):
        start = time.perf_counter()
        func(*args)
        times.append(time.perf_counter() - start)

    return min(times)


sizes = [1_000, 5_000, 10_000, 30_000, 50_000]

cover_times = []
job_times = []

for n in sizes:
    test_intervals = generate_interval_cover_instance(
        n=n,
        target_start=TARGET_START,
        target_end=TARGET_END,
        seed=SEED + n
    )

    t_cover = time_function(
        cover_interval_greedy,
        test_intervals,
        TARGET_START,
        TARGET_END,
        repeats=2
    )
    cover_times.append(t_cover)

    test_jobs = generate_jobs(
        n=n,
        max_deadline=min(10_000, max(100, n // 3)),
        seed=SEED + n
    )

    t_job = time_function(
        schedule_jobs_greedy,
        test_jobs,
        None,
        repeats=2
    )
    job_times.append(t_job)

    print(f"n={n:>6,} | phủ đoạn: {t_cover:.4f}s | xếp lịch: {t_job:.4f}s")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(sizes, cover_times, marker="o", label="Phủ đoạn")
plt.plot(sizes, job_times, marker="s", label="Xếp lịch công việc")
plt.xlabel("Kích thước dữ liệu n")
plt.ylabel("Thời gian chạy tốt nhất trong các lần lặp (giây)")
plt.title("Thời gian chạy theo kích thước dữ liệu")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 9. So sánh với hàm `sorted()` của Python

Trong thực tế, `sorted()` của Python sử dụng Timsort, được tối ưu rất mạnh. Vì vậy, tự cài đặt Merge Sort thường chậm hơn.

Tuy nhiên, mục tiêu của bài Lab không phải thay thế `sorted()`, mà là giúp sinh viên thấy rõ:

- sắp xếp là bước tiền xử lý quan trọng;
- sắp xếp có thể được xây dựng bằng chia để trị;
- sau khi sắp xếp, nhiều chiến lược tham lam trở nên đơn giản và hiệu quả.

In [ ]:
def cover_interval_greedy_builtin_sort(intervals, target_start, target_end):
    sorted_intervals = sorted(intervals, key=lambda it: (it.start, -it.end))

    selected = []
    current = target_start
    i = 0
    n = len(sorted_intervals)

    while current < target_end:
        best_interval = None
        best_end = current

        while i < n and sorted_intervals[i].start <= current:
            if sorted_intervals[i].end > best_end:
                best_interval = sorted_intervals[i]
                best_end = sorted_intervals[i].end
            i += 1

        if best_interval is None:
            return None

        selected.append(best_interval)
        current = best_end

    return selected


comparison_intervals = generate_interval_cover_instance(
    n=50_000,
    target_start=TARGET_START,
    target_end=TARGET_END,
    seed=SEED + 999
)

t_merge = time_function(
    cover_interval_greedy,
    comparison_intervals,
    TARGET_START,
    TARGET_END,
    repeats=2
)

t_builtin = time_function(
    cover_interval_greedy_builtin_sort,
    comparison_intervals,
    TARGET_START,
    TARGET_END,
    repeats=2
)

print(f"Merge Sort tự cài đặt: {t_merge:.4f}s")
print(f"sorted() của Python  : {t_builtin:.4f}s")
print("Tỷ lệ thời gian Merge Sort / sorted():", round(t_merge / t_builtin, 2))

## 10. Tổng kết

Bài Lab này minh họa một thông điệp quan trọng:

> Một thuật toán thực tế thường không chỉ thuộc một kỹ thuật duy nhất.

Trong hai bài toán đã xét:

- **chia để trị** xuất hiện qua bước sắp xếp dữ liệu;
- **tham lam** xuất hiện qua bước chọn khoảng hoặc chọn công việc;
- dữ liệu lớn giúp thấy rõ vai trò của độ phức tạp $O(n \log n)$.

Bảng tóm tắt:

| Bài toán | Bước chia để trị | Bước tham lam | Độ phức tạp chính |
|---|---|---|---|
| Phủ đoạn bằng ít khoảng nhất | Sắp xếp khoảng theo điểm bắt đầu | Chọn khoảng vươn xa nhất | $O(n \log n)$ |
| Xếp lịch deadline - lợi nhuận | Sắp xếp công việc theo lợi nhuận | Gán vào slot muộn nhất còn trống | $O(n \log n)$ |

Trong các thuật toán này, nếu bỏ bước sắp xếp, việc ra quyết định tham lam sẽ khó kiểm soát và có thể kém hiệu quả.

## 11. Bài tập

### Bài tập 1. Thay đổi kích thước dữ liệu

Thay đổi:

```python
N_INTERVALS = 100_000
N_JOBS = 100_000
```

Chạy lại các thuật toán và ghi nhận thời gian chạy.

Yêu cầu:

- lập bảng thời gian chạy;
- nhận xét khi kích thước dữ liệu tăng;
- so sánh với dự đoán $O(n \log n)$.

---

### Bài tập 2. Tạo dữ liệu không phủ được đoạn

Sửa hàm sinh khoảng để tạo ra một khoảng trống, ví dụ không có khoảng nào phủ qua vùng $[4000, 4500]$.

Yêu cầu:

- chạy lại `cover_interval_greedy`;
- kiểm tra thuật toán có phát hiện không thể phủ đoạn hay không;
- in ra vị trí `current` tại thời điểm thất bại.

---

### Bài tập 3. So sánh Merge Sort tự cài đặt và `sorted()`

Với các kích thước:

```python
[10_000, 50_000, 100_000, 200_000]
```

hãy so sánh thời gian chạy của:

- `merge_sort_by_key`;
- `sorted()` của Python.

Viết nhận xét: vì sao thư viện chuẩn thường nhanh hơn cài đặt thủ công?

---

### Bài tập 4. Thay đổi phân phối lợi nhuận

Trong bài toán xếp lịch công việc, thay vì sinh lợi nhuận đều trong đoạn `[10, 1000]`, hãy thử:

- nhiều công việc lợi nhuận thấp, ít công việc lợi nhuận rất cao;
- lợi nhuận tỷ lệ nghịch với deadline;
- lợi nhuận tỷ lệ thuận với deadline.

Nhận xét xem số công việc được chọn và tổng lợi nhuận thay đổi thế nào.

---

### Bài tập 5. Khi nào tham lam không còn đúng?

Biến thể bài toán xếp lịch:

- mỗi công việc có thời lượng khác nhau;
- một công việc có thể chiếm nhiều slot;
- lợi nhuận phụ thuộc vào thời điểm hoàn thành.

Hãy giải thích vì sao chiến lược tham lam hiện tại có thể không còn tối ưu.

---

### Bài tập 6. Bài toán tổng hợp tự thiết kế

Tự thiết kế một bài toán thực tế có cấu trúc:

1. cần sắp xếp dữ liệu trước;
2. sau đó dùng chiến lược tham lam để chọn nghiệm.

Gợi ý:

- chọn quảng cáo theo deadline và ngân sách;
- chọn chuyến xe giao hàng để phủ các khu vực;
- chọn lịch bảo trì thiết bị;
- chọn các phiên tư vấn sinh viên trong một ngày.

Yêu cầu nộp:

- phát biểu bài toán;
- tiêu chí sắp xếp;
- chiến lược tham lam;
- phân tích độ phức tạp;
- ví dụ minh họa nhỏ.

## 12. Câu hỏi thảo luận

1. Trong bài toán phủ đoạn, vì sao chọn khoảng có điểm bắt đầu sớm nhất chưa chắc tốt?
2. Trong bài toán phủ đoạn, vì sao chọn khoảng ngắn nhất thường sai?
3. Trong bài toán xếp lịch công việc, vì sao nên đặt công việc vào slot muộn nhất có thể?
4. Bước sắp xếp có phải lúc nào cũng là chia để trị không?
5. Nếu dữ liệu đã được sắp xếp sẵn, độ phức tạp của hai thuật toán còn lại là bao nhiêu?
6. Trong các bài toán thực tế, tiêu chí tham lam thường được tìm bằng cách nào?